# Exploring topics with BERTopic

BERTopic creates an embedding for each text, reduces those vectors, groups similar texts, and finds words to describe each group. We read the original excerpts before naming the groups.

Run the cells from top to bottom. This notebook uses the same BBC sample and excerpts as LDA. The categories are kept for evaluation and never passed to the model. Stability and changing the descriptive words are optional extensions at the end.

`encoder` is the pretrained sentence model. `topic_model` is the BERTopic model we fit. All outputs have been cleared for the new dataset.

The main hands-on task in section 6 asks you to name every topic, build a labelled article DataFrame, and describe and plot the results. Complete it before the BONUS archive analysis.

Before reading and naming topics, section 3 starts with BERTopic's default grouping, then tries finer/coarser grouping and an optional different encoder. Select the final model there before continuing.

In [ ]:
# Hide this known notebook-library warning; keep other warnings visible.
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)

### Install packages

In Colab, run the next cell before the imports. In a local notebook, uncomment it if packages are missing. If installation asks for a restart, restart the runtime and continue with the encoder cell. No Drive mount or manual article upload is needed.

In [ ]:
# %pip install sentence-transformers bertopic gensim pandas matplotlib

### After installation or a runtime restart

Run the warning filter below before continuing. It is repeated here so it also applies if you restarted after installing packages. It hides only this specific `jupyter_client` deprecation warning; other warnings remain visible.

**Already printed warnings remain in the output.** Running the filter again does not erase them. Clear the affected cell's output (or clear all outputs), then rerun the affected cells after this filter. After a runtime restart, rerun this filter and the exercise cells because the previous filter is no longer active.

In [ ]:
# Hide this known notebook-library warning; keep other warnings visible.
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)

### First, inspect a sentence encoder

The encoder reuses pretrained parameters to return one numeric vector per sentence. We do not train it on these articles. The shape below means **(number of sentences, dimensions per sentence)**. Individual dimensions do not correspond to named words.

The English MiniLM encoder is the default. To try an alternative, comment out the active assignment and uncomment exactly one other `model_name` line. Rerun all later embedding and fitting cells after changing it. Model cards: [English MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), [MPNet](https://huggingface.co/sentence-transformers/all-mpnet-base-v2), [multilingual MiniLM](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), [lecture encoder](https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2).

The first run downloads the encoder; a CPU is sufficient for this sample.

In [ ]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
# Alternatives: uncomment ONE and comment out the default above.
# model_name = "sentence-transformers/all-mpnet-base-v2"  # English
# model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# model_name = "sentence-transformers/distiluse-base-multilingual-cased-v2"  # Lecture

encoder = SentenceTransformer(model_name)
sentences = ["This is an example sentence.", "Each sentence becomes a vector."]
sentence_vectors = encoder.encode(sentences)
print("Embedding shape:", sentence_vectors.shape)
print("First five values:", sentence_vectors[0][:5])

### Compare word counts and embeddings

Predict which headline is closest to the first: the same price rise in different words, or the headline sharing “city” and “bus”? Both comparisons use cosine similarity; higher means more similar under that representation.

Add “City lowers bus fares” and rerun. Similar wording or subject does not guarantee the same direction of change or stance.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

headlines = [
    "City raises bus fares",
    "Public transport gets more expensive",
    "City expands bus routes",
]
headline_counts = CountVectorizer().fit_transform(headlines)
headline_vectors = encoder.encode(headlines)

print("Counts:", cosine_similarity(headline_counts[0:1], headline_counts[1:]))
print("Embeddings:", cosine_similarity(headline_vectors[0:1], headline_vectors[1:]))

## 1. Download and sample the BBC news dataset

We use the [BBC News dataset provided by Derek Greene and Pádraig Cunningham](https://github.com/derekgreene/bbc-datasets): 2,225 articles from 2004–2005, in business, entertainment, politics, sport and technology. The source provides it for non-commercial and research purposes; article copyright belongs to the BBC.

The next cell downloads a zip file and reads its text files directly. **No Google Drive mount, Canvas archive or manual upload is needed.** Internet access is needed for the download. Each row will contain an article ID, its original category, a title and the full text. We keep the category for a later check; the model will receive only text.

The `with` blocks close the download and zip when finished. `split("/")` reads the category from a path such as `bbc/sport/001.txt`. `decode` converts the file's bytes into text.

In [ ]:
from urllib.request import urlopen
from io import BytesIO
from zipfile import ZipFile
import pandas as pd

url = (
    "https://raw.githubusercontent.com/derekgreene/"
    "bbc-datasets/main/raw/bbc-fulltext.zip"
)

with urlopen(url, timeout=60) as response:
    downloaded_file = response.read()

rows = []
with ZipFile(BytesIO(downloaded_file)) as archive:
    for filename in sorted(archive.namelist()):
        parts = filename.split("/")
        if len(parts) == 3 and filename.endswith(".txt"):
            text = archive.read(filename).decode("utf-8-sig").strip()
            lines = text.splitlines()
            rows.append({
                "article_id": filename,
                "category": parts[1],
                "title": lines[0],
                "body": " ".join(lines[1:]),
                "full_text": text,
            })

articles = pd.DataFrame(rows)
print("Articles downloaded:", len(articles))

### Use the same 500 articles in both notebooks

Choose 100 articles from each category. A fixed seed makes the selection repeatable; sorting article IDs gives the same final order in both notebooks. Use the same settings in LDA and BERTopic.

For each article, we use its **title plus the first 100 words of its body**. This keeps the exercise manageable and reduces truncation by the sentence encoder. We are modelling article openings, not every paragraph. The full text remains available for reading.

The balanced sample makes broad subjects easier to compare, but it does not represent the original category proportions or all news coverage. Categories guide our sampling and later evaluation; they are never passed to either model.

In [ ]:
SEED = 42
ARTICLES_PER_CATEGORY = 100
OPENING_WORDS = 100

samples = []
for category, group in articles.groupby("category"):
    chosen = group.sample(n=ARTICLES_PER_CATEGORY, random_state=SEED)
    samples.append(chosen)

sample = pd.concat(samples)
sample = sample.sort_values("article_id").reset_index(drop=True)

# Build the exact text that both models will use.
docs = []
for title, body in zip(sample["title"], sample["body"]):
    opening = " ".join(body.split()[:OPENING_WORDS])
    docs.append(title + "\n\n" + opening)

print("Articles in our sample:", len(docs))
print("First article IDs:", sample["article_id"].head().tolist())
print("\nFirst model input:\n", docs[0])

## 2. Prepare BERTopic's descriptive vocabulary

Keep `docs` unchanged: the encoder reads the same original excerpts as LDA. There is no separate manual lowercasing, punctuation removal or tokenisation of the encoder input.

`CountVectorizer` handles lowercasing, word splitting and stopword removal **for the words used to describe the groups**. We use the same stopword choices as the LDA notebook. The extra words “said”, “mr”, “mrs” and “ms” are a documented teaching choice, not a universal rule.

**Link to the slides:** this is the same document–term count idea as “A document-term matrix with CountVectorizer”. The vectorizer supplies counts for BERTopic's descriptions. Later, we create Gensim's dictionary/count format only for the coherence calculation; those counts do not decide which articles are grouped together.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

news_stopwords = list(ENGLISH_STOP_WORDS) + ["said", "mr", "mrs", "ms"]
vectorizer = CountVectorizer(stop_words=news_stopwords)

### Check the encoder limit before fitting

Both analyses use the same titles and first `OPENING_WORDS` body words, in the same article order. Encoder limits count **model tokens**, not words. This check counts tokens without truncation and stops if any excerpt is too long.

If it stops, reduce `OPENING_WORDS` in **both the LDA and BERTopic notebooks**, then rerun sampling and all later analysis in both. Keep the same seed and article IDs. Do not silently truncate only BERTopic's input or change only one notebook. Repeat the check after changing the encoder.

With the default English MiniLM encoder and this 500-article sample, the 100-word openings fit. Another encoder can have a shorter limit.

In [ ]:
token_lengths = []
for document in docs:
    token_ids = encoder.tokenizer(document, truncation=False)["input_ids"]
    token_lengths.append(len(token_ids))

too_long = sum(length > encoder.max_seq_length for length in token_lengths)
print("Encoder limit:", encoder.max_seq_length, "tokens")
print("Excerpts exceeding the limit:", too_long, "of", len(docs))

if too_long > 0:
    raise ValueError("Reduce OPENING_WORDS in BOTH notebooks, then rerun both analyses.")

## 3. Start with a simple default BERTopic model

### 🟦 Model 1: default grouping

Start with **BERTopic's default grouping settings**. We pass the count vectorizer directly when creating the model, so it uses the news stopwords introduced above for its descriptive words. No extra topic-update step is needed. We supply the embeddings from the encoder introduced above so they can be reused, and fix UMAP's random seed for this classroom comparison.

Read the resulting table before making changes. How many topics appeared? How many articles are unassigned (`-1`)? Are generic words prominent in the automatic descriptions?

Later cells change the level of detail or the encoder, compare the results, and choose a final model. We do not start with a deliberately coarse two-topic solution.

In [ ]:
from bertopic import BERTopic

embeddings_1 = encoder.encode(docs, show_progress_bar=True)

model_1 = BERTopic(vectorizer_model=CountVectorizer(stop_words=news_stopwords))
model_1.umap_model.set_params(random_state=SEED)  # Reproducible classroom run.
model_1.fit_transform(docs, embeddings_1)
model_1.get_topic_info()

### 🟧 Model 2: change how fine-grained the topics are

Now repeat the fit with **`min_topic_size=5`** to allow smaller groups. The default is 10. **Edit that one number and rerun this cell:** try 5 for finer detail or 30 for broader topics. Smaller values do not guarantee more useful topics, and the unassigned percentage can also change.

Keep the same articles, embeddings, seed and vectorizer settings. `min_topic_size` is BERTopic's beginner-friendly control for HDBSCAN's minimum cluster size; with the default clustering setup, changing it can also affect outlier decisions. It does not specify an exact number of topics.

The finer version is the starting choice for the hands-on exercise below, rather than a deliberately coarse two-topic example. Check its actual topic count in your run.

In [ ]:
# CHANGE HERE: 5 allows finer groups; try 30 for a coarser comparison.
model_2 = BERTopic(min_topic_size=5, vectorizer_model=CountVectorizer(stop_words=news_stopwords))
model_2.umap_model.set_params(random_state=SEED)
model_2.fit_transform(docs, embeddings_1)
model_2.get_topic_info()

### 🟪 Optional model 3: change the encoder

The first two models use the same encoder. This optional version uses [English MPNet](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) instead. It needs a separate download and new embeddings, so it takes longer.

Change `alternative_model_name` if you want to try another sentence encoder. The length check stops if any excerpt would be truncated: shorten `OPENING_WORDS` in **both** notebooks and rerun all analyses in that case.

This version keeps the default grouping settings of `model_1` and the same descriptive vectorizer. Compare it with `model_1` to investigate the encoder change. Skip this cell if you only want the first two models, and leave its comparison-table line commented out.

In [ ]:
alternative_model_name = "sentence-transformers/all-mpnet-base-v2"
alternative_encoder = SentenceTransformer(alternative_model_name)

alternative_lengths = []
for document in docs:
    token_ids = alternative_encoder.tokenizer(document, truncation=False)["input_ids"]
    alternative_lengths.append(len(token_ids))
if max(alternative_lengths) > alternative_encoder.max_seq_length:
    raise ValueError("Reduce OPENING_WORDS in BOTH notebooks and rerun both analyses.")

embeddings_3 = alternative_encoder.encode(docs, show_progress_bar=True)
model_3 = BERTopic(vectorizer_model=CountVectorizer(stop_words=news_stopwords))
model_3.umap_model.set_params(random_state=SEED)
model_3.fit_transform(docs, embeddings_3)
model_3.get_topic_info()

### Compare versions before choosing

Rerun the table after changing and refitting a model. Compare:

- **Assigned topics:** how many groups were found, excluding `-1`?
- **Unassigned percentage:** how much of the sample was left outside those groups?
- **Mean coherence:** do five leading words per group co-occur in the original excerpts?
- **Topic words and original texts:** are the groups meaningful and distinguishable?

All coherence calculations use the same original dictionary, counts and five-word rule. The table reports how many topics were eligible for scoring. **Higher `u_mass` is better (−2 is higher than −10), with no universal cut-off.** A model can score well while leaving many articles unassigned or producing broad, generic groups. Use all these checks when choosing.

To include the optional encoder experiment, run it first and uncomment its line in the comparison cell. Compare model 2 with model 1 for granularity, and model 3 with model 1 for the encoder change.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# Reference counts for coherence only; identical processing to the LDA notebook.
analyze = vectorizer.build_analyzer()
tokens = []
for document in docs:
    tokens.append(analyze(document))
dictionary = Dictionary(tokens)
corpus = []
for words in tokens:
    corpus.append(dictionary.doc2bow(words))

comparison_rows = []
for name, fitted_model in [
    ("model_1: defaults", model_1),
    ("model_2: finer groups", model_2),
    # ("model_3: different encoder", model_3),  # Uncomment after the optional fit.
]:
    word_lists = []
    assigned_topics = 0
    for topic_id in fitted_model.get_topics():
        if topic_id != -1:
            assigned_topics += 1
            words = [word for word, weight in fitted_model.get_topic(topic_id)[:5]]
            if len(words) == 5 and all(word in dictionary.token2id for word in words):
                word_lists.append(words)

    score = float("nan")  # No score if no group has five eligible words.
    if word_lists:
        score = CoherenceModel(
            topics=word_lists, corpus=corpus, dictionary=dictionary,
            coherence="u_mass",
        ).get_coherence()

    comparison_rows.append({
        "Model": name,
        "Assigned topics": assigned_topics,
        "Topics scored": len(word_lists),
        "Unassigned (%)": 100 * fitted_model.topics_.count(-1) / len(docs),
        "Mean coherence (u_mass)": score,
    })

model_comparison = pd.DataFrame(comparison_rows)
model_comparison.round(3)

### 🟦 Choose a final model and continue with enough topics to inspect

We start the hands-on exercise with **`model_2`**, the finer-grained version. Check the topic count and the topic words in your run; choose the version that gives useful distinctions, not just the highest coherence score. The default `model_1` remains available for comparison.

Choose the matching pair of lines below. Models 1 and 2 use `embeddings_1`; optional model 3 uses `embeddings_3`. The selected model supplies the groups, names, DataFrame and visualisations below.

If your chosen model gives only one or two broad topics, try a smaller `min_topic_size` in model 2, rerun that fit and the comparison, then select again. An exact topic count is not guaranteed. After changing the final choice, rerun the cells from here to the BONUS heading and revise the naming dictionary.

In [ ]:
# Default hands-on choice: the finer-grained version.
topic_model = model_2
embeddings = embeddings_1

# Alternative: use the default model.
# topic_model = model_1
# embeddings = embeddings_1

# Alternative: only after running the optional encoder example.
# topic_model = model_3
# embeddings = embeddings_3

groups = topic_model.topics_
strengths = topic_model.probabilities_
reducer = topic_model.umap_model
vectorizer = topic_model.vectorizer_model
topic_table = topic_model.get_topic_info()
print("Assigned topics:", sum(topic_table["Topic"] != -1))
topic_table

## 4. Read the groups and original excerpts

`Topic` is a group ID, `Count` is its number of articles, and `Name` is an automatic word-based name. IDs do not give a quality ranking. The `-1` row collects unassigned articles, which need not share a subject.

Use the final model selected above. Choose an assigned topic ID from its table. Read three excerpts, propose a short label and look for an ambiguous example. Change `selected_topic` to `-1` to inspect unassigned material. To read a complete source article, use `print(sample.loc[article_id, "full_text"])`.

Term weights from `get_topic` are descriptive weights, not word counts or probabilities.

In [ ]:
# Keep leading terms from assigned groups for the checks below.
topic_words = {}
for topic_id in topic_table["Topic"]:
    if topic_id != -1:
        topic_words[topic_id] = [word for word, weight in topic_model.get_topic(topic_id)[:5]]

if not topic_words:
    raise ValueError("No assigned topics found. Inspect -1 and try different clustering settings.")

selected_topic = list(topic_words.keys())[0]  # Or enter another ID from the table.
print("Top words:", topic_model.get_topic(selected_topic))

The next cell randomly selects up to three matching excerpts. This gives another view than reading only the first matching article. Three examples are a starting point; read more before accepting a label.

In [ ]:
matching_articles = []
for article_id, group in enumerate(groups):
    if group == selected_topic:
        matching_articles.append(article_id)

import random
chosen_articles = random.Random(SEED).sample(
    matching_articles, k=min(3, len(matching_articles))
)
for article_id in chosen_articles:
    print(f"\nArticle {article_id}\n{docs[article_id]}")

unassigned = groups.count(-1)
print(f"\nUnassigned articles: {unassigned}/{len(groups)} ({unassigned / len(groups):.0%})")

## 5. Check coherence

Coherence asks whether the leading words tend to occur together in the original excerpts. We use Gensim's `u_mass` and five leading terms per topic, with the same reference counts in both notebooks.

**Higher is better: −2 is better than −10.** More strongly negative values mean weaker co-occurrence under this measure. Scores are commonly negative because the calculation uses log probabilities. There is no universal passing score. This is different from perplexity, where lower values indicate better predictive fit under comparable conditions.

The table and optional dot plot show individual topics, so a model-wide average does not hide weak ones. Points farther right have higher coherence. Read one lower-scoring and one higher-scoring topic before judging their usefulness. A high score does not establish that a human label fits.

[Gensim coherence documentation](https://radimrehurek.com/gensim/models/coherencemodel.html)

As in the model comparison, only topics with five eligible terms are scored. The printed coverage shows how many topics were included; smaller groups can have too few usable terms. Inspect the unscored groups separately rather than treating the mean as covering every topic.


In [ ]:
from gensim.models import CoherenceModel

# Use the same five-word eligibility rule as the model comparison.
scored_words = {}
for topic_id, words in topic_words.items():
    if len(words) == 5 and all(word in dictionary.token2id for word in words):
        scored_words[topic_id] = words
print("Topics scored:", len(scored_words), "of", len(topic_words))
if not scored_words:
    raise ValueError("No topic has five eligible words. Inspect the topic descriptions first.")

coherence = CoherenceModel(
    topics=list(scored_words.values()), corpus=corpus,
    dictionary=dictionary, coherence="u_mass",
)
coherence_table = pd.DataFrame({
    "Topic": list(scored_words.keys()),
    "Top words": [", ".join(words) for words in scored_words.values()],
    "Coherence (u_mass)": coherence.get_coherence_per_topic(),
})
print("Mean coherence:", round(coherence_table["Coherence (u_mass)"].mean(), 3))
coherence_table

In [ ]:
import matplotlib.pyplot as plt

ordered = coherence_table.sort_values("Coherence (u_mass)")
plt.figure(figsize=(8, 4))
plt.scatter(ordered["Coherence (u_mass)"], range(len(ordered)))
plt.yticks(range(len(ordered)), ordered["Topic"])
plt.xlabel("Coherence (u_mass): higher is better →")
plt.ylabel("Topic ID")
plt.title("Which topics should we inspect more closely?")
plt.tight_layout()
plt.show()

## 6. Hands-on: name the topics and build your results DataFrame

**Your task:** inspect the topics, record your names in one editable dictionary, then attach those names to the articles. Complete this before the BONUS section.

1. Inspect the topic overview below. It lists each ID, leading words and three example article row numbers.
2. Read the example excerpts. Choose a short, specific name and note one supporting passage and one ambiguous example.
3. Edit the example names in the **Assign your names** dictionary below to match what you found. Keep the numeric topic IDs as keys. Give different IDs distinct names so summaries stay easy to interpret.
4. Run the mapping, DataFrame and plotting cells. If you revise a name, edit the same dictionary and rerun those cells.

This keeps all naming decisions visible and reproducible in the notebook, without interactive prompts. Name topics before looking at the original news categories. Names do not change fitted topics. After refitting, inspect the new IDs and revise the mapping rather than reusing old names automatically.

In [ ]:
topic_rows = []
for topic_id, words in topic_words.items():
    examples = []
    for article_id, group in enumerate(groups):
        if group == topic_id:
            examples.append(article_id)
    topic_rows.append({
        "Topic ID": topic_id,
        "Leading words": ", ".join(words),
        "Example article rows": examples[:3],
    })

topic_overview = pd.DataFrame(topic_rows)
display(topic_overview)

for row in topic_rows:
    print(f"\nTOPIC {row['Topic ID']}: {row['Leading words']}")
    for article_id in row["Example article rows"]:
        print(f"\nArticle {article_id}: {docs[article_id]}")

### Assign your names in one place

Edit the example dictionary below directly. The number on the left is a topic ID; the text on the right is your name for it. **These are example names, not verified labels for your model.** Compare each ID with the overview and original excerpts, then replace its name. Add or remove entries if your model found a different number of topics.

After changing a name, rerun the mapping, DataFrame and plotting cells below. The mapping cell reports missing names or IDs that are not in your current model. BERTopic keeps `-1` as “Unassigned”.

**Do this manually:** change the strings in `topic_names` below, for example `0: "Your own name"`. “Example topic 1”, “Example topic 2”, etc. are placeholders only. Check the topic overview and add entries for any further IDs. Then rerun the mapping, `results` and plotting cells: your names become the labels in those tables and figures.

In [ ]:
# Placeholder names only: replace these after reading your model's topics.
# This template covers the 14 topics found in the demonstrated finer-grained run.
# Add or remove entries if your current model has different IDs.
topic_names = {
    0: "Example topic 1",
    1: "Example topic 2",
    2: "Example topic 3",
    3: "Example topic 4",
    4: "Example topic 5",
    5: "Example topic 6",
    6: "Example topic 7",
    7: "Example topic 8",
    8: "Example topic 9",
    9: "Example topic 10",
    10: "Example topic 11",
    11: "Example topic 12",
    12: "Example topic 13",
    13: "Example topic 14",
}

In [ ]:
# Apply your names and report anything that still needs attention.
topic_labels = {}
for topic_id in topic_words:
    label = topic_names.get(topic_id, "").strip()
    if label == "":
        label = f"Topic {topic_id} (not named yet)"
        print(f"Please add a name for topic {topic_id} in topic_names.")
    topic_labels[topic_id] = label

for topic_id in topic_names:
    if topic_id not in topic_words:
        print(f"Topic {topic_id} is not in this model. Check your dictionary.")

# Keep unassigned articles separate from named subjects.
topic_labels[-1] = "Unassigned"
topic_model.set_topic_labels(topic_labels)
topic_labels

### Attach the model output to the articles

`results` has **one row per article**, with its original ID, title, model excerpt and original category. We add its group ID, your topic name and the clustering membership strength. `map(topic_labels)` looks up your name for each numeric ID.

The order of `groups` and `strengths` matches `docs` and `sample`: do not reorder just one object before this step. Membership strength describes the clustering assignment; it is not an LDA topic share or a probability that your human label is correct. Keep unassigned articles visible in your summaries.

In [ ]:
results = sample[["article_id", "title", "category"]].copy()
results["text"] = docs
results["topic_id"] = groups
results["topic_label"] = results["topic_id"].map(topic_labels)
results["membership_strength"] = strengths

results.head()

### Describe and plot your named topics

Count how many articles have each topic name and express that count as a percentage of the sample. `value_counts()` counts rows; dividing by `len(results)` gives a share of articles.

**Try it:** which named topic is largest? How many articles and what percentage does it contain? Choose one topic, read more of its articles using the filter below, and write two sentences describing what it covers and where its label falls short.

In [ ]:
topic_counts = results["topic_label"].value_counts()
summary = topic_counts.rename("Articles").to_frame()
summary["Percent of articles"] = (100 * summary["Articles"] / len(results)).round(1)
display(summary)

topic_counts.sort_values().plot.barh(figsize=(9, 5))
plt.xlabel("Number of articles")
plt.ylabel("Your topic name")
plt.title("Articles by named topic")
plt.tight_layout()
plt.show()

In [ ]:
# Replace this with another name from your summary table.
label_to_read = results["topic_label"].iloc[0]
chosen_articles = results.loc[results["topic_label"] == label_to_read]
print("Articles with this label:", len(chosen_articles))

for article_id in chosen_articles.head(3).index:
    print(f"\n{results.loc[article_id, 'title']}")
    print(results.loc[article_id, "text"])

### Finish the hands-on task

Before continuing, make sure every assigned topic has your own name. Then write:

1. The name, count and percentage of the largest topic.
2. A short description of one topic, supported by an excerpt and an ambiguous example.
3. One limitation of the bar chart. For LDA, compare counts with mean topic shares; for BERTopic, report how many articles are unassigned.

Your `results` DataFrame is now ready for ordinary pandas filtering, grouping and plotting. If you want a CSV, uncomment the export below. It writes to your current notebook folder (the temporary session storage in Colab), not automatically to Google Drive. If you share it, include the BBC dataset attribution and source-use conditions from section 1.

In [ ]:
# Optional: save your labelled results for later use.
# results.to_csv("named_topic_results.csv", index=False)

## 7. Compare with the original news categories

Now that you have named the topics yourself, reveal the original five news categories as a reference. They were not supplied to the model. The table counts articles from each original category within each model group. A group can span categories, and a category can contain several meaningful subtopics.

Do not assume that a topic model must reproduce the five editorial categories exactly. Read examples to decide whether an unexpected grouping is useful. For LDA, this table uses each article's largest topic share as a simplified summary; the article still has a mixture.

In [ ]:
# Compare your own names with the original editorial categories.
pd.crosstab(results["topic_label"], results["category"])

## 8. More ways to inspect your results

Try these views after naming your topics. You do not have to use every plot. For each one, choose something you notice and return to the underlying articles to check your interpretation. Give the plots your own descriptive titles when reporting results.

### Category heatmap: what is inside each named topic?

Each row is one of your topic names; columns are the original BBC categories. Each row sums to 100%. A dark cell means many articles **within that topic** belong to that category. This normalisation makes differently sized topics easier to compare, but it hides their absolute sizes: use the count table alongside it.

This is a comparison with editorial categories, not a topic-similarity matrix. A mixed row might reveal a useful cross-category theme or a poorly separated topic. Read examples to distinguish these explanations. For LDA, the row uses each article's largest-share topic and does not show its full mixture.

In [ ]:
category_percentages = pd.crosstab(
    results["topic_label"], results["category"], normalize="index"
) * 100

display(category_percentages.round(1))
plt.figure(figsize=(10, 5))
plt.imshow(category_percentages, aspect="auto", cmap="Blues", vmin=0, vmax=100)
plt.xticks(range(len(category_percentages.columns)), category_percentages.columns,
           rotation=45, ha="right")
plt.yticks(range(len(category_percentages.index)), category_percentages.index)
plt.colorbar(label="Percent of articles within this topic")
plt.xlabel("Original news category")
plt.ylabel("Your topic name")
plt.title("Which news categories make up each topic?")
plt.tight_layout()
plt.show()

### Interactive topic words and intertopic map

Each panel shows a topic's leading words and their descriptive weights, not probabilities. Use the chart to choose groups for document reading. The optional intertopic map uses one point per topic, not per article; a two-dimensional projection can lose information.

In [ ]:
topic_model.visualize_barchart(top_n_topics=min(5, len(topic_words)), custom_labels=True)

In [ ]:
# The map requires enough assigned topics for its projection.
if len(topic_words) >= 4:
    display(topic_model.visualize_topics(custom_labels=True))
else:
    print("Too few assigned topics for this map; use the word chart and original excerpts.")

### Interactive document map: inspect articles near each other

**Each point is an article**, unlike the intertopic map where each point is a topic. Hover to read the excerpt; use the legend to show or hide groups. Colours and labels refer to the groups already fitted, not a new clustering.

We reuse the embeddings, but creating this two-dimensional projection takes extra time. Its axes have no named substantive meaning, and nearby points in 2D are only an approximation of the relationships in the original vectors. Use the map to choose articles for reading rather than to establish topic validity.

In [ ]:
document_map = topic_model.visualize_documents(
    docs,
    embeddings=embeddings,
    custom_labels=True,
    hide_annotations=True,
)
document_map

### Interactive similarity heatmap: which topics look related?

This heatmap compares **topic representations with each other**. It is different from the category heatmap above. Larger similarities suggest related representations; a diagonal cell compares a topic with itself.

Use it to select two related topics and check whether your names distinguish their articles meaningfully. Similarity does not establish that they should be merged.

In [ ]:
if len(topic_words) >= 2:
    display(topic_model.visualize_heatmap(custom_labels=True))
else:
    print("Need at least two assigned topics for a similarity comparison.")

### Interactive hierarchy: explore broader and narrower groupings

The tree suggests how topic word representations can be joined into larger groups. It does not change the fitted assignments or automatically validate a broader label. Choose one suggested combination, read excerpts from both topics and decide whether a shared name would be useful.

In [ ]:
if len(topic_words) >= 2:
    display(topic_model.visualize_hierarchy(custom_labels=True))
else:
    print("Need at least two assigned topics for a hierarchy.")

## Optional: stability across five random seeds

**Question:** do articles stay together when we rerun the selected BERTopic settings? We keep the **same excerpts, embeddings, clustering settings and descriptive vectorizer**, and change only UMAP's random seed. Embeddings are reused, so the encoder is not downloaded or run again. Five fits take longer than one but leave your selected model, names and results intact.

Five runs are a practical classroom starting point, not a methodological standard. They test random-seed sensitivity, not whether the results generalise to another sample. Check the topic counts and unassigned percentages as well as agreement.

We use adjusted Rand index (ARI), which allows topic IDs to change: **1 means identical groupings, around 0 means chance-level agreement, and negative values indicate below-chance agreement.** There is no universal passing score. A value such as 0.8 indicates considerable agreement, not an automatic quality stamp. [Scikit-learn ARI documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)

In this optional loop, `clone` copies the selected settings into a fresh component for each fit; it does not copy the fitted groups. There is no topic-update step.


In [ ]:
from sklearn.base import clone

stability_seeds = [42, 43, 44, 45, 46]
bertopic_stability_groups = []
bertopic_run_rows = []

for seed in stability_seeds:
    repeated_model = BERTopic(
        umap_model=clone(topic_model.umap_model).set_params(random_state=seed),
        hdbscan_model=clone(topic_model.hdbscan_model),
        vectorizer_model=clone(topic_model.vectorizer_model),
        ctfidf_model=clone(topic_model.ctfidf_model),
    )
    repeated_groups, _ = repeated_model.fit_transform(docs, embeddings)
    bertopic_stability_groups.append(repeated_groups)
    bertopic_run_rows.append({
        "Seed": seed,
        "Assigned topics": len(set(repeated_groups) - {-1}),
        "Unassigned (%)": 100 * repeated_groups.count(-1) / len(docs),
    })

bertopic_stability_runs = pd.DataFrame(bertopic_run_rows)
bertopic_stability_runs.round(1)

### Compare every pair and report coverage

Five runs give ten pairs. For each pair, compare only articles assigned to a group **in both runs**. Otherwise the shared `-1` label could treat unrelated unassigned articles as a substantive group.

The shared subset can differ between pairs: always read its article count and percentage alongside ARI. A high score covering a small part of the sample gives limited reassurance. Pairs with fewer than two shared groups in either run are marked `NaN` rather than assigned a misleading score.

The pairs share fitted models and are not independent experiments. The median and range summarise observed variation, not a confidence interval.

In [ ]:
from itertools import combinations
from sklearn.metrics import adjusted_rand_score

bertopic_pair_rows = []
for first_run, second_run in combinations(range(len(stability_seeds)), 2):
    first_shared = []
    second_shared = []
    for first, second in zip(bertopic_stability_groups[first_run],
                             bertopic_stability_groups[second_run]):
        if first != -1 and second != -1:
            first_shared.append(first)
            second_shared.append(second)

    ari = float("nan")
    if len(set(first_shared)) >= 2 and len(set(second_shared)) >= 2:
        ari = adjusted_rand_score(first_shared, second_shared)
    bertopic_pair_rows.append({
        "First seed": stability_seeds[first_run],
        "Second seed": stability_seeds[second_run],
        "Shared assigned articles": len(first_shared),
        "Shared coverage (%)": 100 * len(first_shared) / len(docs),
        "ARI": ari,
    })

bertopic_stability_pairs = pd.DataFrame(bertopic_pair_rows)
display(bertopic_stability_pairs.round(3))
display(bertopic_stability_pairs["ARI"].agg(["median", "min", "max"]).round(3))
print("Pairs with a usable ARI:", bertopic_stability_pairs["ARI"].notna().sum(), "of 10")

### Example interpretation — one checked run, not a target score

> “Across five seeds, median ARI was approximately 0.28, ranging from 0.07 to almost 1. Runs found between 5 and 25 topics, and shared assigned coverage ranged from about 74% to 100%. Some pairs agreed closely, but the overall solution changed substantially with the seed. I would not treat one high-agreement pair as evidence of a stable topic structure.”

This example indicates seed sensitivity even though two pairs have very high ARI. Broad groups in some runs may split into several finer groups in others; inspect the articles and topic words to see what changes. If an ARI of 0.9 covered only 20% of the articles, the conclusion about the whole corpus would be much weaker. Coarse, uninformative groups can also be highly stable.

**Write your own interpretation:** report the median and range of usable ARIs, shared coverage, topic counts and unassigned percentages. If some pairs have no usable ARI, report that too. Read articles from a changing group and explain whether your substantive conclusion depends on that instability. Do not compare identical topic ID numbers across runs as if they had a fixed meaning.

These numbers came from one checked run of the provided BBC exercise. Your selected model, package versions and computing environment can change the results. Use your own tables when writing your conclusion; these numbers are not acceptance thresholds.


## Optional: change descriptive words, keep the groups

The baseline counts single words. On a copy of the fitted model, also allow two-word phrases. `update_topics` changes the descriptions without rerunning the grouping. Compare a group's terms and decide whether your label changes. This optional copy needs additional memory.

In [ ]:
from copy import deepcopy

phrase_model = deepcopy(topic_model)
phrase_vectorizer = CountVectorizer(stop_words=news_stopwords, ngram_range=(1, 2))
phrase_model.update_topics(docs, vectorizer_model=phrase_vectorizer)

print("Single words:", topic_model.get_topic(selected_topic))
print("Words and phrases:", phrase_model.get_topic(selected_topic))

## Something to think about

1. Propose a label for one topic and quote a passage supporting it. Find an ambiguous example too.
2. Compare a lower- and higher-coherence topic. Do your readings support that ranking?
3. Compare the model groups with the original news categories. What does the category table miss?
4. Compare LDA and BERTopic on the same article IDs. How do mixtures, groups and unassigned articles differ?

Both models use the same sample, excerpts and coherence reference. Their output structures and numbers of topics can still differ. Use several checks instead of choosing a winner from one score.

**Corpus-selection extension:** the earlier political-news archive produced many Trump-related terms. That can reveal a feature of the corpus; nearly identical topic descriptions can also indicate weak separation. Compare this broader-subject dataset with that narrower corpus. Removing names changes the measurement and does not remove the underlying subject.

These BBC files do not provide reliable per-article publication dates. We therefore do not plot topics over time or invent dates from filenames.

Which BERTopic version would you keep? Explain the trade-off between topic words, coherence and unassigned coverage.

## BONUS: the dated Canvas archive and topics over time

**BONUS — optional. Finish naming your topics, building `results` and making the plots above first.** The BBC exercise above is complete. Use this separate extension if you want to analyse the dated news in Canvas `articles.zip`. Keep the BBC results as your baseline; the variables below start with `archive_` so they do not replace it.

**Setup:** use the already extracted `articles` folder. There is no need to zip it again. In Colab, connect Drive below and set `ARCHIVE_PATH` to that folder. On your own computer, put `articles/` next to the notebook or edit the path. The folder should contain dated folders, then outlet folders, then article files. If the folder check prints `False`, copy the correct folder path from Colab’s Files panel and update it before continuing.

The archive contains dated folders and outlet subfolders. We select up to 100 files per outlet using the same seed, then model each title plus the first 100 body words. An equal-outlet sample is not representative of each outlet's publication volume. Filename dates are used as publication-date metadata; check the source documentation and a few original files before interpreting trends. Also check that filename titles match their bodies.

**Run the cells below only if you want to do this bonus and have the extracted `articles` folder ready. Otherwise, stop here.** If using Colab, uncomment the Drive connection lines below and authorize access; then set the folder path to its Drive location.

**Colab folder:** `My Drive/google-drive-colab/bdaca_week5/articles`, mounted as `/content/drive/MyDrive/google-drive-colab/bdaca_week5/articles`. Use the extracted folder.

If you open this local edition in Colab, uncomment the two Drive lines below. Use the `_colab` edition for ready-made installation and Drive cells.

In [ ]:
# Colab only (optional):
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# Point to the extracted articles folder, not to articles.zip.
# Expected layout: articles/<dated folder>/<outlet>/<article file>.
# In Colab: /content/drive/MyDrive/google-drive-colab/bdaca_week5/articles
ARCHIVE_PATH = Path("articles")
print("Folder exists:", ARCHIVE_PATH.is_dir())

In [ ]:
import random

archive_outlets = ["BBC", "CNN", "The Guardian", "Vox", "Infowars"]
archive_rng = random.Random(SEED)
archive_rows = []

for outlet in archive_outlets:
    files = sorted(ARCHIVE_PATH.glob(f"*/{outlet}/*"))
    files = [file for file in files if file.is_file()]
    if not files:
        raise FileNotFoundError(
            f"No articles found for {outlet} under {ARCHIVE_PATH}. "
            "Set ARCHIVE_PATH to the extracted articles folder containing the dated folders."
        )
    chosen_files = archive_rng.sample(files, min(100, len(files)))
    for file in chosen_files:
        source, date, title = file.name.split("--", 2)
        body = file.read_text(encoding="utf-8").strip()
        opening = " ".join(body.split()[:OPENING_WORDS])
        archive_rows.append({
            "date": date, "outlet": outlet,
            "text": title + "\n\n" + opening,
        })

archive_data = pd.DataFrame(archive_rows)
archive_data["month"] = pd.to_datetime(archive_data["date"]).dt.to_period("M").dt.to_timestamp()
archive_docs = archive_data["text"].tolist()
print("Archive excerpts:", len(archive_docs))
display(pd.crosstab(archive_data["month"], archive_data["outlet"]))

### Dynamic BERTopic: topics and words over time

We now implement BERTopic's **topics-over-time** workflow on the dated Canvas articles. We first fit groups on all sampled excerpts, then estimate their words and frequencies within each month.

The groups remain fixed across months; the **word representations vary over time**. This is BERTopic's dynamic topic representation approach, not a separate clustering for each month or a dynamic LDA model. Keeping group IDs fixed lets us follow a topic across the period.

Use the month-by-outlet table above to check coverage. The archive's political focus may produce several topics about the same actor; inspect their issues and original excerpts before naming them. This is a separate model from the BBC exercise.

In [ ]:
# Check the archive excerpts too: do not silently truncate them.
archive_lengths = []
for document in archive_docs:
    archive_lengths.append(len(encoder.tokenizer(document, truncation=False)["input_ids"]))
if max(archive_lengths) > encoder.max_seq_length:
    raise ValueError("Reduce OPENING_WORDS in both archive analyses and reload the excerpts.")

archive_embeddings = encoder.encode(archive_docs, show_progress_bar=True)
archive_model = BERTopic(
    min_topic_size=5,
    vectorizer_model=CountVectorizer(stop_words=news_stopwords),
)
archive_model.umap_model.set_params(random_state=SEED)
archive_groups, _ = archive_model.fit_transform(archive_docs, archive_embeddings)
archive_model.get_topic_info()

### Calculate monthly topic representations

Each row below is a **topic in one month**, with its frequency and descriptive words. We use the archive's monthly timestamps instead of inventing dates for the BBC data.

`evolution_tuning=True` smooths a month's word representation using the previous observed time slice; `global_tuning=True` also blends it with the whole-period topic representation. These are BERTopic's standard tuning options. Set both to `False` if you want unsmoothed month-specific descriptions and compare the `Words` column.

[BERTopic topics-over-time guide](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)

In [ ]:
archive_over_time = archive_model.topics_over_time(
    archive_docs,
    timestamps=archive_data["month"].tolist(),
    evolution_tuning=True,
    global_tuning=True,
)
archive_over_time.head(10)

### Plot topic frequency over time

Hover over the interactive lines to inspect the words representing a topic in each month. The y-axis shows **counts of sampled articles assigned to the topic**, not LDA topic shares. Counts can rise because more articles were sampled that month.

Look for a period when a topic's words or frequency change, then read the corresponding excerpts before interpreting it as a substantive development.

In [ ]:
archive_model.visualize_topics_over_time(archive_over_time, top_n_topics=5)

### Inspect how one topic's words change

Choose a topic ID from the archive model's topic table. The filtered table lets you read the words in chronological order. A group can keep the same broad subject while the events, people or policy details associated with it change.

In [ ]:
# Replace with another assigned topic ID from the archive model's table.
archive_topic_id = 0
archive_over_time.loc[
    archive_over_time["Topic"] == archive_topic_id,
    ["Timestamp", "Frequency", "Words"],
].sort_values("Timestamp")

### Check relative attention as well as counts

This second chart shows the **percentage of sampled articles assigned to each group within a month**. We plot the five largest groups; “Unassigned” appears if it is among them. Percentages use all sampled articles that month, so the displayed lines do not necessarily add up to 100%. It helps separate changing monthly sample sizes from changing relative attention. Percentages are still affected by which outlets and articles were sampled; the corpus is not a census of all news.

Missing months are absent, so lines may bridge gaps. These are assignment percentages, not LDA's estimated mixtures.

In [ ]:
archive_data["topic_id"] = archive_groups
archive_monthly_counts = pd.crosstab(archive_data["month"], archive_data["topic_id"])
archive_monthly_percent = archive_monthly_counts.div(
    archive_monthly_counts.sum(axis=1), axis=0
) * 100
# Plot the five largest groups so the chart stays readable.
archive_top_topics = archive_monthly_counts.sum().nlargest(5).index
archive_percent_to_plot = archive_monthly_percent[archive_top_topics].rename(
    columns={-1: "Unassigned"}
)
archive_percent_to_plot.plot(figsize=(10, 5), marker="o")
plt.xlabel("Publication month")
plt.ylabel("Percent of sampled articles in this month")
plt.title("Relative topic attention in the Canvas archive")
plt.legend(title="Archive topic ID")
plt.tight_layout()
plt.show()

### Discuss the archive results

Do several topics mention Trump? Is the model separating issues such as elections, trade or immigration, or mainly repeating the same actor names? Corpus selection, date range and the sample shape what the model can find. Similar-looking topics require further reading and validation.

For a stricter comparison between LDA and BERTopic on this optional archive, keep sample settings and text construction identical and repeat the encoder-length check on `archive_docs`. The LDA and BERTopic charts above measure different things, so compare interpretations and coverage rather than the heights of their lines.